# Imports and configuration

In [ ]:
!pip install torch-geometric

In [ ]:
# ============================================================================
# GENERACIÓN DE GRAFOS TEMPORALES DINÁMICOS PARA NIDS
# Versión Final con manejo correcto de protocolos y centinelas
# ============================================================================

import pandas as pd
import numpy as np
import pickle
import torch
from torch_geometric.data import Data
from collections import defaultdict
import time
import os
import gc

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

# Paths (AJUSTAR según tu Google Drive)
CSV_PATH = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv'
FILTERED_CHUNKS_FILE = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl'
OUTPUT_DIR = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_DYNAMIC/'
TEMP_DIR = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_DYNAMIC/temp/'

# Crear directorio de salida
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

# Columnas del dataset
TIMESTAMP_COL = 'FLOW_START_MILLISECONDS'
SRC_IP_COL = 'IPV4_SRC_ADDR'
DST_IP_COL = 'IPV4_DST_ADDR'
LABEL_COL = 'Attack'

# Parámetros de ventanas temporales
WINDOW_SIZE = pd.Timedelta(minutes=10)   # Tumbling windows de 10 min
WINDOW_STEP = pd.Timedelta(minutes=10)   # Sin overlap

# Procesamiento
CHUNK_SIZE = 250000

# Números de protocolo IP (IANA standard)
IPPROTO_ICMP = 1      # Internet Control Message Protocol
IPPROTO_TCP = 6       # Transmission Control Protocol
IPPROTO_UDP = 17      # User Datagram Protocol
IPPROTO_ICMPV6 = 58   # ICMP for IPv6

print("="*70)
print("📋 CONFIGURACIÓN")
print("="*70)
print(f"Window size: {WINDOW_SIZE}")
print(f"Window step: {WINDOW_STEP}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Temp dir: {TEMP_DIR} (evita saturar RAM)")
print("="*70)

📋 CONFIGURACIÓN
Window size: 0 days 00:10:00
Window step: 0 days 00:10:00
Output dir: /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_DYNAMIC/
Temp dir: /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_DYNAMIC/temp/ (evita saturar RAM)


In [ ]:
# Cambiar al directorio
os.chdir('/content/drive/MyDrive/nids-mitre/')

# Verificar que estás ahí
!pwd

# Ver qué archivos .py tenés
!ls

# Read chunks

In [ ]:
# ============================================================================
# FUNCIÓN: Leer chunks filtrados
# ============================================================================

def read_filtered_chunks(pickle_file):
    """Lee chunks desde pickle"""
    with open(pickle_file, 'rb') as f:
        while True:
            try:
                yield pickle.load(f)
            except EOFError:
                break

# Node features

In [ ]:
def precompute_ip_flows_ULTRA_FAST(window_df, src_col, dst_col):
    """
    Pre-agrupa flows por IP usando groupby vectorizado
    15-20x más rápido que filtros repetidos
    """

    ip_flows_dict = {}

    # Crear DataFrame largo: (IP, flow_idx, role)
    src_flows = pd.DataFrame({
        'ip': window_df[src_col],
        'flow_idx': window_df.index,
        'role': 'src'
    })

    dst_flows = pd.DataFrame({
        'ip': window_df[dst_col],
        'flow_idx': window_df.index,
        'role': 'dst'
    })

    all_mappings = pd.concat([src_flows, dst_flows], ignore_index=True)

    # Agrupar por IP (UNA SOLA PASADA)
    for ip, group in all_mappings.groupby('ip'):

        flow_indices_src = group[group['role'] == 'src']['flow_idx'].values
        flow_indices_dst = group[group['role'] == 'dst']['flow_idx'].values
        flow_indices_all = group['flow_idx'].unique()

        ip_flows_dict[ip] = {
            'as_src': window_df.loc[flow_indices_src] if len(flow_indices_src) > 0 else pd.DataFrame(),
            'as_dst': window_df.loc[flow_indices_dst] if len(flow_indices_dst) > 0 else pd.DataFrame(),
            'all': window_df.loc[flow_indices_all]
        }

    return ip_flows_dict




In [ ]:
# ============================================================================
# FUNCIÓN: Calcular NODE FEATURES (enriquecidas)
# ============================================================================

def compute_node_features_FAST(ip, ip_flows_dict, src_col, dst_col):
    """
    Versión optimizada - acceso directo sin búsqueda
    """

    flows_data = ip_flows_dict[ip]
    as_src = flows_data['as_src']
    as_dst = flows_data['as_dst']
    all_flows = flows_data['all']

    # Resto IDÉNTICO al original
    total_flows = len(all_flows)
    flows_as_src = len(as_src)
    flows_as_dst = len(as_dst)

    deg_in = as_dst[src_col].nunique() if len(as_dst) > 0 else 0
    deg_out = as_src[dst_col].nunique() if len(as_src) > 0 else 0

    bytes_out = as_src['OUT_BYTES'].sum() if len(as_src) > 0 else 0
    bytes_in = as_dst['IN_BYTES'].sum() if len(as_dst) > 0 else 0
    ratio_out_in = bytes_out / (bytes_in + 1e-6)

    pkts_out = as_src['OUT_PKTS'].sum() if len(as_src) > 0 else 0
    pkts_in = as_dst['IN_PKTS'].sum() if len(as_dst) > 0 else 0

    num_peers = deg_in + deg_out

    ports_src = as_src['L4_SRC_PORT'].nunique() if len(as_src) > 0 and 'L4_SRC_PORT' in as_src.columns else 0
    ports_dst = as_dst['L4_DST_PORT'].nunique() if len(as_dst) > 0 and 'L4_DST_PORT' in as_dst.columns else 0

    if len(all_flows) > 0:
        tcp_ratio = (all_flows['PROTOCOL'] == IPPROTO_TCP).sum() / len(all_flows)
        udp_ratio = (all_flows['PROTOCOL'] == IPPROTO_UDP).sum() / len(all_flows)
        icmp_ratio = ((all_flows['PROTOCOL'] == IPPROTO_ICMP) |
                      (all_flows['PROTOCOL'] == IPPROTO_ICMPV6)).sum() / len(all_flows)
        avg_duration = all_flows['FLOW_DURATION_MILLISECONDS'].mean()
        std_duration = all_flows['FLOW_DURATION_MILLISECONDS'].std()
        if pd.isna(std_duration):
            std_duration = 0.0
    else:
        tcp_ratio = udp_ratio = icmp_ratio = avg_duration = std_duration = 0.0

    return [
        total_flows, flows_as_src, flows_as_dst,
        bytes_out, bytes_in, ratio_out_in,
        pkts_out, pkts_in,
        deg_in, deg_out, num_peers,
        ports_src, ports_dst,
        tcp_ratio, udp_ratio, icmp_ratio,
        avg_duration, std_duration
    ]



# Edge features

In [ ]:
# ============================================================================
# FUNCIÓN: Calcular EDGE FEATURES (enriquecidas con manejo de centinelas)
# ============================================================================

def compute_edge_features(flows_df):
    """Calcula 37 features por edge con manejo de centinelas"""

    n_flows = len(flows_df)

    is_tcp = (flows_df['PROTOCOL'] == IPPROTO_TCP)
    is_udp = (flows_df['PROTOCOL'] == IPPROTO_UDP)
    is_icmp = (flows_df['PROTOCOL'] == IPPROTO_ICMP)
    is_icmpv6 = (flows_df['PROTOCOL'] == IPPROTO_ICMPV6)
    is_dns = (flows_df['DNS_QUERY_TYPE'] != 0) if 'DNS_QUERY_TYPE' in flows_df.columns else pd.Series([False]*len(flows_df))
    has_icmp = is_icmp | is_icmpv6

    # DNS
    dns_flows = flows_df[is_dns]
    if len(dns_flows) > 0 and 'DNS_TTL_ANSWER' in dns_flows.columns:
        dns_ttl_mean = dns_flows['DNS_TTL_ANSWER'].mean()
        dns_ttl_std = dns_flows['DNS_TTL_ANSWER'].std()
        if pd.isna(dns_ttl_std): dns_ttl_std = 0.0
        dns_query_type_mode = dns_flows['DNS_QUERY_TYPE'].mode()[0] if len(dns_flows) > 0 else 0
        edge_has_dns = 1.0
        dns_ratio = len(dns_flows) / n_flows
    else:
        dns_ttl_mean = dns_ttl_std = dns_query_type_mode = edge_has_dns = dns_ratio = 0.0

    # ICMP
    icmp_flows = flows_df[has_icmp]
    if len(icmp_flows) > 0 and 'ICMP_TYPE' in icmp_flows.columns:
        icmp_types = icmp_flows['ICMP_TYPE'] // 256
        icmp_codes = icmp_flows['ICMP_TYPE'] % 256
        icmp_type_mode = icmp_types.mode()[0] if len(icmp_types) > 0 else 0
        icmp_code_mode = icmp_codes.mode()[0] if len(icmp_codes) > 0 else 0
        edge_has_icmp = 1.0
        icmp_ratio = len(icmp_flows) / n_flows
        edge_has_icmpv4 = 1.0 if is_icmp.any() else 0.0
        edge_has_icmpv6 = 1.0 if is_icmpv6.any() else 0.0
    else:
        icmp_type_mode = icmp_code_mode = edge_has_icmp = icmp_ratio = edge_has_icmpv4 = edge_has_icmpv6 = 0.0

    # TCP
    tcp_flows = flows_df[is_tcp]
    if len(tcp_flows) > 0:
        tcp_flags_mode = tcp_flows['TCP_FLAGS'].mode()[0] if 'TCP_FLAGS' in tcp_flows.columns and len(tcp_flows) > 0 else 0
        tcp_win_mean = tcp_flows['TCP_WIN_MAX_IN'].mean() if 'TCP_WIN_MAX_IN' in tcp_flows.columns else 0
        tcp_win_max = tcp_flows['TCP_WIN_MAX_IN'].max() if 'TCP_WIN_MAX_IN' in tcp_flows.columns else 0
        edge_has_tcp = 1.0
        tcp_ratio = len(tcp_flows) / n_flows
    else:
        tcp_flags_mode = tcp_win_mean = tcp_win_max = edge_has_tcp = tcp_ratio = 0.0

    # UDP
    edge_has_udp = 1.0 if is_udp.any() else 0.0
    udp_ratio = is_udp.sum() / n_flows

    # Básicas
    duration_mean = flows_df['FLOW_DURATION_MILLISECONDS'].mean()
    duration_std = flows_df['FLOW_DURATION_MILLISECONDS'].std()
    duration_min = flows_df['FLOW_DURATION_MILLISECONDS'].min()
    duration_max = flows_df['FLOW_DURATION_MILLISECONDS'].max()

    bytes_in_sum = flows_df['IN_BYTES'].sum()
    bytes_out_sum = flows_df['OUT_BYTES'].sum()
    bytes_total = bytes_in_sum + bytes_out_sum
    bytes_ratio = bytes_out_sum / (bytes_in_sum + 1e-6)

    pkts_in_sum = flows_df['IN_PKTS'].sum()
    pkts_out_sum = flows_df['OUT_PKTS'].sum()
    pkts_total = pkts_in_sum + pkts_out_sum
    avg_pkt_size = bytes_total / (pkts_total + 1e-6)

    # Opcionales
    iat_src_dst_mean = flows_df['SRC_TO_DST_IAT_AVG'].mean() if 'SRC_TO_DST_IAT_AVG' in flows_df.columns else 0
    iat_src_dst_std = flows_df['SRC_TO_DST_IAT_STDDEV'].mean() if 'SRC_TO_DST_IAT_STDDEV' in flows_df.columns else 0
    iat_dst_src_mean = flows_df['DST_TO_SRC_IAT_AVG'].mean() if 'DST_TO_SRC_IAT_AVG' in flows_df.columns else 0
    iat_dst_src_std = flows_df['DST_TO_SRC_IAT_STDDEV'].mean() if 'DST_TO_SRC_IAT_STDDEV' in flows_df.columns else 0

    ttl_min = flows_df['MIN_TTL'].min() if 'MIN_TTL' in flows_df.columns else 0
    ttl_max = flows_df['MAX_TTL'].max() if 'MAX_TTL' in flows_df.columns else 0
    ttl_mean = flows_df['MIN_TTL'].mean() if 'MIN_TTL' in flows_df.columns else 0

    retrans_in = flows_df['RETRANSMITTED_IN_PKTS'].sum() if 'RETRANSMITTED_IN_PKTS' in flows_df.columns else 0
    retrans_out = flows_df['RETRANSMITTED_OUT_PKTS'].sum() if 'RETRANSMITTED_OUT_PKTS' in flows_df.columns else 0

    def safe(x):
        return 0.0 if pd.isna(x) or np.isnan(x) or np.isinf(x) else float(x)

    return [
        safe(bytes_in_sum), safe(bytes_out_sum), safe(bytes_ratio),
        safe(pkts_in_sum), safe(pkts_out_sum), safe(avg_pkt_size),
        safe(duration_mean), safe(duration_std), safe(duration_min), safe(duration_max),
        safe(retrans_in), safe(retrans_out),
        safe(iat_src_dst_mean), safe(iat_src_dst_std),
        safe(iat_dst_src_mean), safe(iat_dst_src_std),
        safe(ttl_min), safe(ttl_max), safe(ttl_mean),
        safe(dns_ttl_mean), safe(dns_ttl_std), safe(dns_query_type_mode),
        safe(icmp_type_mode), safe(icmp_code_mode),
        safe(tcp_flags_mode), safe(tcp_win_mean), safe(tcp_win_max),
        edge_has_tcp, edge_has_udp, edge_has_icmpv4, edge_has_icmpv6, edge_has_dns,
        tcp_ratio, udp_ratio, icmp_ratio, dns_ratio,
        n_flows
    ]

# Dynamic graphs

In [ ]:
# ============================================================================
# FUNCIÓN: Crear grafo con NODOS DINÁMICOS + Trazabilidad
# ============================================================================

def create_graph_dynamic_FAST(window_df, window_id, window_start, window_end,
                              src_col, dst_col, label_col, global_ip_to_id):
    """
    Versión optimizada con pre-agrupación
    """

    # IPs activas
    active_ips = set(window_df[src_col].unique()) | set(window_df[dst_col].unique())
    active_ips = sorted(list(active_ips))

    if len(active_ips) == 0:
        return None

    local_ip_to_idx = {ip: idx for idx, ip in enumerate(active_ips)}
    local_to_global = {idx: global_ip_to_id[ip] for idx, ip in enumerate(active_ips)}

    # ← CAMBIO CRÍTICO: Usar versión ULTRA_FAST
    ip_flows_dict = precompute_ip_flows_ULTRA_FAST(window_df, src_col, dst_col)

    # Node features (ahora SÍ será rápido)
    node_features = []
    for ip in active_ips:
        features = compute_node_features_FAST(ip, ip_flows_dict, src_col, dst_col)
        node_features.append(features)

    # Edges (sin cambios)
    edge_list = []
    edge_features = []
    edge_to_flows = {}
    edge_to_labels = {}

    for (src_ip, dst_ip), flows_df in window_df.groupby([src_col, dst_col]):
        src_idx = local_ip_to_idx[src_ip]
        dst_idx = local_ip_to_idx[dst_ip]

        edge_list.append([src_idx, dst_idx])
        edge_id = len(edge_list) - 1

        agg_features = compute_edge_features(flows_df)
        edge_features.append(agg_features)

        edge_to_flows[edge_id] = flows_df.index.tolist() if hasattr(flows_df, 'index') else list(range(len(flows_df)))
        edge_to_labels[edge_id] = flows_df[label_col].tolist()

    # Label
    has_attack = (window_df[label_col] != 0).any()
    y = torch.tensor([1 if has_attack else 0], dtype=torch.long)

    if has_attack:
        attack_flows = (window_df[label_col] != 0).sum()
        attack_ratio = attack_flows / len(window_df)
        attack_types = window_df[window_df[label_col] != 0][label_col].unique().tolist()
    else:
        attack_flows = attack_ratio = 0
        attack_types = []

    if len(edge_list) == 0:
        return None

    data = Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_list, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_features, dtype=torch.float),
        y=y,
        num_nodes=len(active_ips)
    )

    data.window_id = window_id
    data.window_start = window_start
    data.window_end = window_end
    data.num_flows = len(window_df)
    data.attack_flows = attack_flows
    data.attack_ratio = attack_ratio
    data.attack_types = attack_types
    data.local_to_global = local_to_global
    data.edge_to_flows = edge_to_flows
    data.edge_to_labels = edge_to_labels

    return data


# Pipeline

In [ ]:
# ============================================================================
# PIPELINE PRINCIPAL: Generación de grafos (CON VERBOSE DETALLADO)
# ============================================================================

def generate_graphs_optimized():
    """Pipeline con nodos dinámicos + disco"""

    print("\n" + "="*70)
    print("🚀 GENERACIÓN OPTIMIZADA")
    print("="*70)

    # FASE 1: Mapeo IPs
    print("\n1️⃣  Mapeo global...")
    start_time = time.time()

    global_ip_to_id = {}
    next_id = 0

    for i, chunk in enumerate(read_filtered_chunks(FILTERED_CHUNKS_FILE)):
        for ip in pd.concat([chunk[SRC_IP_COL], chunk[DST_IP_COL]]).unique():
            if ip not in global_ip_to_id:
                global_ip_to_id[ip] = next_id
                next_id += 1
        print(f"   Chunk {i+1}: {len(global_ip_to_id):,} IPs")
        del chunk
        gc.collect()

    print(f"\n   ✅ {time.time()-start_time:.1f}s | {len(global_ip_to_id):,} IPs")

    with open(f'{OUTPUT_DIR}/global_ip_to_id.pkl', 'wb') as f:
        pickle.dump(global_ip_to_id, f)

    # FASE 2: Rango temporal
    print("\n2️⃣  Rango temporal...")
    start_time = time.time()

    dataset_start = dataset_end = None

    for chunk in read_filtered_chunks(FILTERED_CHUNKS_FILE):
        chunk[TIMESTAMP_COL] = pd.to_datetime(chunk[TIMESTAMP_COL].astype('int64') * 1e6, unit='ns')
        chunk_start = chunk[TIMESTAMP_COL].min()
        chunk_end = chunk[TIMESTAMP_COL].max()
        if dataset_start is None or chunk_start < dataset_start: dataset_start = chunk_start
        if dataset_end is None or chunk_end > dataset_end: dataset_end = chunk_end
        del chunk
        gc.collect()

    print(f"   ✅ {time.time()-start_time:.1f}s | {dataset_start} → {dataset_end}")

    # FASE 3: Escribir ventanas a DISCO
    print("\n3️⃣  Escribiendo ventanas a disco (evita saturar RAM)...")
    start_time = time.time()

    window_files = defaultdict(list)

    for i, chunk in enumerate(read_filtered_chunks(FILTERED_CHUNKS_FILE)):
        chunk[TIMESTAMP_COL] = pd.to_datetime(chunk[TIMESTAMP_COL].astype('int64') * 1e6, unit='ns')
        chunk['window_id'] = ((chunk[TIMESTAMP_COL] - dataset_start) // WINDOW_STEP).astype(int)

        for wid in chunk['window_id'].unique():
            window_chunk = chunk[chunk['window_id'] == wid]
            temp_file = f'{TEMP_DIR}/w{wid:04d}_c{i:03d}.pkl'
            with open(temp_file, 'wb') as f:
                pickle.dump(window_chunk, f)
            window_files[wid].append(temp_file)

        print(f"   Chunk {i+1}: {len(window_files)} ventanas")
        del chunk
        gc.collect()

    print(f"\n   ✅ {(time.time()-start_time)/60:.1f}m | {len(window_files)} ventanas")

    # FASE 4: Crear grafos
    print("\n4️⃣  Creando grafos (nodos dinámicos)...")
    start_time = time.time()

    graphs_metadata = []
    skipped = 0

    with open(f'{OUTPUT_DIR}/graphs_dynamic.pkl', 'wb') as f_out:
        for i, wid in enumerate(sorted(window_files.keys())):

            # Leer de disco
            chunks = [pickle.load(open(f, 'rb')) for f in window_files[wid]]
            window_df = pd.concat(chunks, ignore_index=True)
            del chunks
            gc.collect()

            window_start = dataset_start + wid * WINDOW_STEP
            window_end = window_start + WINDOW_SIZE

            window_df = window_df[(window_df[TIMESTAMP_COL] >= window_start) &
                                 (window_df[TIMESTAMP_COL] < window_end)]

            if len(window_df) < 5:
                skipped += 1
                continue

            # CAMBIO A VERSION FAST
            graph = create_graph_dynamic_FAST(window_df, wid, window_start, window_end,
                                        SRC_IP_COL, DST_IP_COL, LABEL_COL, global_ip_to_id)

            if graph is not None:
                pickle.dump(graph, f_out)
                graphs_metadata.append({
                    'window_id': wid,
                    'label': graph.y.item(),
                    'num_nodes': graph.num_nodes,
                    'num_edges': graph.edge_index.shape[1]
                })

            if (i+1) % 10 == 0:
                elapsed = time.time() - start_time
                est = (elapsed/(i+1)) * (len(window_files)-(i+1))
                print(f"   {i+1}/{len(window_files)} | {elapsed/60:.1f}m | ~{est/60:.1f}m restante")

            del window_df, graph
            gc.collect()

    print(f"   ✅ {(time.time()-start_time)/60:.1f}m | {len(graphs_metadata)} grafos")

    # Limpiar
    import shutil
    shutil.rmtree(TEMP_DIR)

    # Stats
    stats_df = pd.DataFrame(graphs_metadata)
    print(f"\n✅ Completado: {len(graphs_metadata)} grafos")
    print(f"   Normal: {(stats_df['label']==0).sum()} | Ataque: {(stats_df['label']==1).sum()}")
    print(f"   Avg nodos: {stats_df['num_nodes'].mean():.0f}")

    return graphs_metadata



In [ ]:
# ============================================================================
# EJECUTAR GENERACIÓN
# ============================================================================

if __name__ == '__main__':

    # Verificar que archivos necesarios existen
    if not os.path.exists(FILTERED_CHUNKS_FILE):
        print(f"❌ ERROR: No se encuentra {FILTERED_CHUNKS_FILE}")
        print("   Primero debes crear filtered_chunks.pkl")
    else:
        # Ejecutar
        start_time = time.time()

        graphs_metadata = generate_graphs_optimized()

        elapsed = time.time() - start_time
        print(f"\n⏱️  Tiempo total: {elapsed/60:.1f} minutos")

        print("\n✅ PROCESO COMPLETADO")


🚀 GENERACIÓN OPTIMIZADA

1️⃣  Mapeo global...
   Chunk 1: 9,578 IPs
   Chunk 2: 13,914 IPs
   Chunk 3: 16,299 IPs
   Chunk 4: 18,227 IPs
   Chunk 5: 22,371 IPs
   Chunk 6: 25,800 IPs
   Chunk 7: 28,007 IPs
   Chunk 8: 31,309 IPs
   Chunk 9: 36,022 IPs
   Chunk 10: 39,184 IPs
   Chunk 11: 42,331 IPs
   Chunk 12: 45,172 IPs
   Chunk 13: 48,652 IPs
   Chunk 14: 51,391 IPs
   Chunk 15: 54,118 IPs
   Chunk 16: 57,907 IPs
   Chunk 17: 60,440 IPs
   Chunk 18: 62,389 IPs
   Chunk 19: 65,451 IPs
   Chunk 20: 67,023 IPs
   Chunk 21: 69,304 IPs
   Chunk 22: 71,639 IPs
   Chunk 23: 76,575 IPs
   Chunk 24: 79,351 IPs
   Chunk 25: 80,634 IPs
   Chunk 26: 83,008 IPs
   Chunk 27: 86,463 IPs
   Chunk 28: 89,562 IPs
   Chunk 29: 92,189 IPs
   Chunk 30: 95,668 IPs
   Chunk 31: 99,099 IPs
   Chunk 32: 102,260 IPs
   Chunk 33: 105,517 IPs
   Chunk 34: 107,977 IPs
   Chunk 35: 108,287 IPs
   Chunk 36: 108,569 IPs
   Chunk 37: 108,914 IPs
   Chunk 38: 109,317 IPs
   Chunk 39: 110,308 IPs
   Chunk 40: 112,54